# 04 三维胚胎点云
寻找含 x/y/z 的本地坐标表并用 Plotly 绘制。无真实坐标时使用合成测试点，仅验证交互绘图和文件输出。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tables = list((project/'data/processed/coordinates').glob('*.csv')) + list((project/'data/processed/coordinates').glob('*.tsv'))
real = False
for table in tables:
    frame = pd.read_csv(table, sep='\t' if table.suffix == '.tsv' else ',')
    lower = {c.lower(): c for c in frame.columns}
    if all(k in lower for k in ('x','y','z')):
        points = frame.rename(columns={lower['x']:'x', lower['y']:'y', lower['z']:'z'})
        real = True
        break
if not real:
    rng = np.random.default_rng(20240322)
    z = rng.uniform(-1, 1, 1800)
    angle = rng.uniform(0, 2*np.pi, 1800)
    r = np.sqrt(np.clip(1-z*z, 0, None))*rng.uniform(.65, 1, 1800)
    points = pd.DataFrame({'x': 1.7*r*np.cos(angle), 'y': .8*r*np.sin(angle), 'z': z})
points.shape, ('真实本地坐标' if real else '合成测试数据（非论文结果）')

D:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


((1800, 3), '合成测试数据（非论文结果）')

In [2]:
sample = points.sample(min(len(points), 50000), random_state=1)
title = 'Local 3D coordinates' if real else 'Synthetic test point cloud (not a paper result)'
fig = px.scatter_3d(sample, x='x', y='y', z='z', color='z', opacity=.55, title=title)
fig.update_traces(marker={'size': 2})
out = project/'results/figures'/('coordinates_3d.html' if real else 'synthetic_test_coordinates_3d.html')
fig.write_html(out, include_plotlyjs='cdn')
out

WindowsPath('C:/Users/l/Documents/Codex/2026-08-15/github/spatial-omics-literature-data/papers/xiao-2024-human-gastrulation/results/figures/synthetic_test_coordinates_3d.html')